### Half-open Intervals

The following Python functions are from the course notes:

In [ ]:
def runwasm(wasmfile):
  from IPython.display import display, Javascript

  with open(wasmfile, 'rb') as file:
    wasm_bytes = file.read()  # read the WASM file and convert its content to a comma-separated string of byte values
    wasm_byte_array_str = ','.join(str(byte) for byte in wasm_bytes)

  display(
    Javascript(f""" // Javascript code to compile and run the WASM module
    const wasmBinary = new Uint8Array([{wasm_byte_array_str}]); // convert back to binary representation
 
    const params = {{
        'P0lib': {{
            'write': i => element.append(i + ' '),
            'writeln': () => element.append(document.createElement('br')),
            'read': () => window.prompt()
        }}
    }};
 
    WebAssembly.compile(wasmBinary.buffer) // compile shareable code
        .then(module => WebAssembly.instantiate(module, params)) // create an instance with memory
        // .then(instance => instance.exports.program()); // run the main program; not needed if a start function is specified
    """)
  )

In [ ]:
def runpywasm(wasmfile):
  from pywasm.core import Machine, Runtime, FuncType, ValType

  # P0lib implementation in Python
  def write(_: Machine, args: list[int]) -> list[int]:
    print(args[0])
    return []

  def writeln(_: Machine, args: list[int]) -> list[int]:
    print()
    return []

  def read(_: Machine, args: list[int]) -> list[int]:
    return [int(input())]

  # Create runtime
  runtime = Runtime()
  runtime.imports = {
    'P0lib': {
      'write': runtime.allocate_func_host(FuncType([ValType.i32()], []), write),
      'writeln': runtime.allocate_func_host(FuncType([], []), writeln),
      'read': runtime.allocate_func_host(FuncType([], [ValType.i32()]), read),
    }
  }
  # Create and run instance
  instance = runtime.instance_from_file(wasmfile)

In [ ]:
def runwasmtime(wasmfile):
  from wasmtime import Engine, Store, Module, Linker, Func, FuncType, ValType

  # P0lib implementation in Python
  def write(i: int):
    print(i)

  def writeln():
    print()

  def read() -> int:
    return int(input())

  # Create engine, store, and module
  engine = Engine()
  store = Store(engine)
  module = Module(store.engine, open(wasmfile, 'rb').read())
  # Use Linker to create the P0lib library
  write_func = Func(store, FuncType([ValType.i32()], []), write)
  writeln_func = Func(store, FuncType([], []), writeln)
  read_func = Func(store, FuncType([], [ValType.i32()]), read)
  linker = Linker(engine)
  linker.define(store, 'P0lib', 'write', write_func)
  linker.define(store, 'P0lib', 'writeln', writeln_func)
  linker.define(store, 'P0lib', 'read', read_func)
  # Create and run an instance
  instance = linker.instantiate(store, module)

In [ ]:
import import_ipynb
from P0 import compileString

In this version of P0, arrays are declared using an interval-like notation for the domain. Array declarations are often of the form that the upper bound has `- 1`, as in:

In [ ]:
compileString(
  """
const N = 4
var a: [0 .. N - 1] → integer

procedure index(x: integer) → (i: integer)
    i := 0
    while (i < N) and (a[i] ≠ x) do i := i + 1

program arithmetic
    var x: integer
      a[0] := 3; a[1] := 5; a[2] := 7; a[3] := 9
      x ← read(); x ← index(x); write(x)
""",
  'search.wat',
)

Modify the P0 compiler that comes with this lab to allow intervals of the form `[a .. b)`, with the meaning `[a .. b - 1]`. State which parts of the compiler need to be modified. Give the modifications to the grammar in the next cell. A test case is below.

**Which parts of the compiler need to be modified:**

Only the **parser** (`P0.ipynb`) needs to be modified, specifically the `typ()` procedure. The scanner (`SC.ipynb`) already recognizes `)` as `RPAREN`, so no scanner changes are needed. The symbol table (`ST.ipynb`) is unaffected since `Array(base, lower, length)` remains the same representation. The code generator (`CGwat.ipynb`) is unaffected since `genArray` receives the same `Array` object regardless of whether the interval was closed or half-open.

**Grammar modification for `type`:**

The original production:

    type ::= ident | "[" expression ".." expression "]" "→" type | "(" typedIds ")" | "set" "[" expression ".." expression "]"

becomes:

    type ::= ident | "[" expression ".." expression ( "]" | ")" ) "→" type | "(" typedIds ")" | "set" "[" expression ".." expression "]"

The only change is that the closing bracket after the upper bound can be either `"]"` (closed interval `[a .. b]`, length = b − a + 1) or `")"` (half-open interval `[a .. b)`, length = b − a).

**Modification to `typ()` in `P0.ipynb`:** After parsing the upper bound expression, check whether the closing delimiter is `RBRAK` or `RPAREN`:

```python
if SC.sym == RBRAK:
    getSym(); halfopen = False
elif SC.sym == RPAREN:
    getSym(); halfopen = True
else: mark("']' or ')' expected")
...
length = y.val - x.val if halfopen else y.val - x.val + 1
x = Type(CG.genArray(Array(z, x.val, length)))
```

In [ ]:
compileString(
  """
const N = 4
var a: [0 .. N) → integer

procedure index(x: integer) → (i: integer)
    i := 0
    while (i < N) and (a[i] ≠ x) do i := i + 1

program arithmetic
    var x: integer
      a[0] := 3; a[1] := 5; a[2] := 7; a[3] := 9
      x ← read(); x ← index(x); write(x)
""",
  'search.wat',
)

In [ ]:
!wat2wasm search.wat

In [ ]:
runwasm('search.wasm')

In [ ]:
runpywasm('search.wasm')

In [ ]:
runwasmtime('search.wasm')